# Operations on Arrays and Dictionaries

{doc}`./placeholders` and {doc}`./decision_variables` explained how to define arrays and dictionaries of different variable types.
JijModeling can also work with arrays and dictionaries containing general elements, not only variables. We refer to these collectively as **collections**.
For a detailed conceptual explanation of arrays and dictionaries and guidance on choosing between them, see [Arrays and dictionaries of variables](#containers_of_vars) in {doc}`./variables`.

This chapter first reviews the different kinds of collections, then explains the functions used to generate them and how to access their elements.
The next chapter, {doc}`./stream_and_logical_ops`, also explains how to treat arrays and dictionaries as streams and take their sums and products.

In [1]:
import jijmodeling as jm

(generators)=
## Generating Collections: {py:func}`~jijmodeling.genarray` and {py:func}`~jijmodeling.gendict`

As described in {doc}`./placeholders` and {doc}`./decision_variables`, arrays and dictionaries can be introduced when declaring variables. They can also be generated from other expressions.
Use {py:func}`~jijmodeling.genarray` to generate an array and {py:func}`~jijmodeling.gendict` to generate a dictionary.

### Array Generation with {py:func}`~jijmodeling.genarray`

{py:func}`~jijmodeling.genarray` is similar to NumPy's {py:func}`~numpy.fromfunction`[^numpy-fromfunction]. It generates a new array from a shape and a function (the generator function) that maps indices to elements.
The following example uses {py:func}`genarray <jijmodeling.genarray>` to generate an array of shape $(N, M)$ whose elements are the sums of their respective indices:

[^numpy-fromfunction]: Depending on the form of its generator function, NumPy's {py:func}`~numpy.fromfunction` may generate an array with a shape different from the one supplied, or even a scalar. JijModeling's {py:func}`~jijmodeling.genarray` is guaranteed to return an array with the supplied shape regardless of the generator function's form.

In [2]:
problem = jm.Problem("Array and Dict Example")
N = problem.Length("N")
M = problem.Length("M")

jm.genarray(lambda i, j: i + j, (N, M))

Expression(genarray(lambda (i, j): i + j, (N, M)))

Within the Decorator API, you can write the same expression concisely with a comprehension:

In [3]:
@problem.update
def _(problem: jm.DecoratedProblem):
    display(jm.genarray(i + j for (i, j) in (N, M)))

Expression(genarray(lambda (i, j): i + j, (N, M)))

Here, the tuple `(N, M)` on the right-hand side of `in` is shorthand for the Cartesian product of `N` and `M`.
This notation is explained further in {doc}`stream_and_logical_ops`.

Comprehensions passed to {py:func}`~jijmodeling.genarray` support exactly one `for` clause and do not support `if` clauses.
For example, using multiple `for` clauses as follows results in an error:

In [4]:
try:

    @jm.Problem.define("genarray example")
    def problem(problem):
        N = problem.Natural()
        M = problem.Natural()
        a = problem.Float(shape=(N, M))
        x = problem.BinaryVar(shape=N)
        Sums = problem.NamedExpr(jm.genarray(a[i, j] * x[i] for i in N for j in M))

except SyntaxError as e:
    print(str(e))

error[E-SE0002] A `genarray` comprehension must have exactly one for-clause.

Possible fix: use a single `for` that iterates over the whole shape or key set at once.

File "/home/h_ishii_j_ij_com/.cache/tmp/ipykernel_2670921/1800573765.py", line 9, col 46-82:

    9  |          Sums = problem.NamedExpr(jm.genarray(a[i, j] * x[i] for i in N for j in M))
                                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

Hint: You can read the description and possible fix at https://jij-inc-jijmodeling.readthedocs-hosted.com/en/stable/error_codes/error/E-SE0002.html


### Dictionary Generation with {py:func}`~jijmodeling.gendict`

The dictionary generator {py:func}`~jijmodeling.gendict` creates a new dictionary from an expression representing a key set and a generator function that maps keys to values.
Because the generator function always returns a value, a dictionary generated by {py:func}`~jijmodeling.gendict` is always a `TotalDict`.

The following example generates a dictionary keyed by a category label $L$ and a natural number $N$:

In [5]:
problem = jm.Problem("Array and Dict Example")
N = problem.Natural("N")
L = problem.CategoryLabel("L")
x = problem.BinaryVar("x", dict_keys=L)
jm.gendict(lambda l, n: x[l] + n, (L, N))

Expression(gendict(lambda (l, n): x[l] + n, ((L, N))))

In the Decorator API, {py:func}`~jijmodeling.gendict` also supports comprehensions containing exactly one `for` clause and any number of `if` clauses.

In [6]:
@problem.update
def _(problem: jm.DecoratedProblem):
    display(jm.gendict(x[l] + n for (l, n) in (L, N) if n % 2 == 0))

Expression(gendict(lambda (l, n): x[l] + n, (stream((L, N)).filter(lambda (l, n): n % 2 == 0))))

## Obtaining the Domains of Arrays and Dictionaries

For {py:class}`~jijmodeling.Placeholder` and {py:class}`~jijmodeling.DecisionVar` objects, the tuple representing an array's shape is available through the `shape` attribute. For general expressions, use the {py:meth}`Expression.shape() <jijmodeling.Expression.shape>` method.
The expression representing a dictionary's key set is available through {py:meth}`Expression.keys() <jijmodeling.Expression.keys>`. Array expressions also provide {py:meth}`Expression.len_at(n) <jijmodeling.Expression.len_at>` for obtaining the size of the $n$-th dimension.
These operations are useful when formulating mathematical models, for example when defining sums or constraints that iterate over a domain.

## Element Access and Slicing with Indices

As with Python's built-in lists and dictionaries or {py:class}`numpy.ndarray`, elements of JijModeling collections can be accessed with multidimensional indices such as `x[i, j]`.
Specifically, JijModeling supports indexing expressions of the following types:

1. Multidimensional arrays
   + **Allowed indices**: natural-number expressions that contain no decision variables
2. Dictionaries
   + **Allowed indices**: expressions that contain no decision variables and match the dictionary's key type; these can be integers, strings, category labels, or tuples composed of them
3. Tuples
   + **Allowed indices**: natural-number expressions that contain no decision variables and are within the tuple's length

In every case, an index cannot contain a decision variable.
The following example accesses elements of an array and a dictionary by index:

In [7]:
import jijmodeling as jm


@jm.Problem.define("Array and Dict Example")
def problem(problem: jm.DecoratedProblem):
    N = problem.Natural()
    L = problem.CategoryLabel()

    w = problem.Float(shape=N)  # N-element array
    x = problem.BinaryVar(dict_keys=(N, L))  # Dictionary

    problem += jm.sum(w[i] * x[i, l] for i in N for l in L)


problem

Problem(name="Array and Dict Example", sense=MINIMIZE, objective=sum(stream(N.flat_map(lambda (i: natural): L.map(lambda (l: CategoryLabel("L")): (i, l)))).map(lambda ((i, l): Tuple[natural, CategoryLabel("L")]): w[i] * x[i, l])), constraints=[])

Multiple components can be specified at once, as in `x[i,j,k]`. However, using more indices than the number of tuple components, array dimensions, or dictionary key components results in a type error, as shown below.

In [8]:
import jijmodeling as jm


@jm.Problem.define("Array and Dict Example, oversubscripted")
def problem(problem: jm.DecoratedProblem):
    N = problem.Natural()
    M = problem.Natural()

    w = problem.Float(shape=(N, M))  # N × M array

    try:
        problem += jm.sum(w[i, j, i] for i in N for j in M)  # ERROR: too many indices
    except Exception as e:
        print(e)

Traceback (most recent last):
    while checking if expression `sum(stream(N.flat_map(lambda i: M.map(lambda j: (i, j)))).map(lambda (i, j): w[i, j, i]))` has type `float!`,
        defined at File "/home/h_ishii_j_ij_com/.cache/tmp/ipykernel_2670921/3614808236.py", line 12, col 20-60
    while inferring the type of expression `sum(stream(N.flat_map(lambda i: M.map(lambda j: (i, j)))).map(lambda (i, j): w[i, j, i]))`,
        defined at File "/home/h_ishii_j_ij_com/.cache/tmp/ipykernel_2670921/3614808236.py", line 12, col 20-60
    while inferring the type of expression `sum(stream(N.flat_map(lambda i: M.map(lambda j: (i, j)))).map(lambda (i, j): w[i, j, i]))`,
        defined at File "/home/h_ishii_j_ij_com/.cache/tmp/ipykernel_2670921/3614808236.py", line 12, col 20-60
    while inferring the type of expression `stream(N.flat_map(lambda i: M.map(lambda j: (i, j)))).map(lambda (i, j): w[i, j, i])`,
        defined at File "/home/h_ishii_j_ij_com/.cache/tmp/ipykernel_2670921/3614808236

Array indices also support slice notation such as `x[:, 1]`.

In [9]:
import jijmodeling as jm


@jm.Problem.define("Slicing example")
def problem(problem: jm.DecoratedProblem):
    N = problem.Natural()
    M = problem.Natural()

    w = problem.Integer(shape=N)  # N-element array
    x = problem.BinaryVar(shape=(N, M))  # N × M array

    problem += problem.Constraint("sum-per-n", [x[i, :].sum() == w[i] for i in N])


problem

Problem(name="Slicing example", sense=MINIMIZE, objective=0, constraints={sum-per-n: [Constraint(name="sum-per-n", lambda i: sum(x[i, :]) == w[i], domain=stream(N)),],})

Slices that specify a step or stop index, such as `x[1, 1:N:2]`, are also supported.
For details on slice syntax, see the Python docs on "{external+python:ref}`slicings`".